# 21. 언어 지원 분포 분석 — 무반응 그룹 vs 첫 반응군

**분석 목적:**  
무반응 그룹(리뷰 0~9개)과 첫 반응군(리뷰 10~49개)의 지원 언어 수 분포를 비교하여,  
언어 지원 범위가 첫 반응 달성과 연관이 있는지 파악한다.

**사용 데이터:**
- `data/preprocessed/steam_indie_games_silence.csv` — 무반응 그룹 (리뷰 0~9개)
- `data/preprocessed/steam_indie_games_graded.csv` — 반응군 전체 (리뷰 10개 이상)
- `data/raw/public_steam_app_details.csv` — 언어 지원 정보 (`supported_languages`)

**분석 관점:**: 무반응 그룹의 언어 지원 분포: 첫 반응군의 언어 지원 분포: 두 그룹 비교 + 카이제곱 검정 (전략 섹션 근거)

In [1]:
import re
import warnings

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency

warnings.filterwarnings('ignore')

LANG_BINS   = [0, 1, 4, float('inf')]
LANG_LABELS = ['1개 (영어 단독)', '2~4개', '5개 이상']

COLOR_SILENCE  = '#6B7280'
COLOR_RESPONSE = '#4C72B0'
COLORS_GROUP   = ['#9DB8D2', '#4C72B0', '#1E3A5F']

## 1. 데이터 로드 및 언어 파싱

In [2]:
def count_languages(lang_str: str) -> int:
    """Steam supported_languages HTML 문자열에서 지원 언어 수를 반환한다."""
    if pd.isna(lang_str):
        return 0
    s = re.sub(r'<br\s*/?>', ', ', lang_str)   # <br> → 쉼표 구분자
    s = re.sub(r'<[^>]+>', '', s)               # 나머지 HTML 태그 제거
    parts = [p.strip().strip('* ') for p in s.split(',')]
    langs = [p for p in parts if p and len(p) > 1 and 'languages with' not in p.lower()]
    return len(langs)

In [3]:
df_silence = pd.read_csv('../../data/preprocessed/steam_indie_games_silence.csv')
df_graded  = pd.read_csv('../../data/preprocessed/steam_indie_games_graded.csv')
df_details = pd.read_csv('../../data/raw/public_steam_app_details.csv')[['appid', 'supported_languages']]

# 첫 반응군: 리뷰 10~49개
df_first = df_graded[df_graded['total_reviews'].between(10, 49)].copy()

print(f'무반응 그룹   : {len(df_silence):,}개 (리뷰 0~9개)')
print(f'첫 반응군     : {len(df_first):,}개 (리뷰 10~49개)')

무반응 그룹   : 6,676개 (리뷰 0~9개)
첫 반응군     : 4,840개 (리뷰 10~49개)


In [4]:
def attach_language(df: pd.DataFrame) -> pd.DataFrame:
    merged = df.merge(df_details, on='appid', how='left')
    merged['lang_count'] = merged['supported_languages'].apply(count_languages)
    merged['lang_group'] = pd.cut(
        merged['lang_count'],
        bins=LANG_BINS,
        labels=LANG_LABELS,
        right=True,
    )
    return merged

df_silence = attach_language(df_silence)
df_first   = attach_language(df_first)

print('무반응 그룹 언어 수 기초 통계:')
print(df_silence['lang_count'].describe().round(2))
print()
print('첫 반응군 언어 수 기초 통계:')
print(df_first['lang_count'].describe().round(2))

무반응 그룹 언어 수 기초 통계:
count    6676.00
mean        3.81
std         8.46
min         0.00
25%         1.00
50%         1.00
75%         3.00
max       103.00
Name: lang_count, dtype: float64

첫 반응군 언어 수 기초 통계:
count    4840.00
mean        4.74
std         8.69
min         0.00
25%         1.00
50%         2.00
75%         6.00
max       103.00
Name: lang_count, dtype: float64


## 2. 무반응 그룹 언어 지원 분포

In [5]:
silence_dist = (
    df_silence.groupby('lang_group', observed=True)
    .agg(game_count=('appid', 'count'))
    .reset_index()
)
silence_dist['ratio'] = silence_dist['game_count'] / silence_dist['game_count'].sum() * 100

display(
    silence_dist.rename(columns={
        'lang_group': '언어 지원 구간',
        'game_count': '게임 수',
        'ratio': '비율 (%)'
    }).round(1).set_index('언어 지원 구간')
)

pct_1  = silence_dist.loc[silence_dist['lang_group'] == '1개 (영어 단독)', 'ratio'].values[0]
pct_5p = silence_dist.loc[silence_dist['lang_group'] == '5개 이상', 'ratio'].values[0]
print(f'\n영어 단독 비율  : {pct_1:.1f}%')
print(f'5개 이상 비율   : {pct_5p:.1f}%')

,게임 수,비율 (%)
언어 지원 구간,,
1개 (영어 단독),3662,56.7
2~4개,1514,23.4
5개 이상,1287,19.9



영어 단독 비율  : 56.7%
5개 이상 비율   : 19.9%


**해석:** 무반응 그룹 중 영어 단독 지원 게임이 56.7%로 절반 이상이다. 5개 이상 언어를 지원하는 게임은 19.9%에 그쳐, 무반응 게임은 노출 기회가 처음부터 제한된 경우가 많음을 확인할 수 있다.

In [6]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=silence_dist['lang_group'].astype(str),
    y=silence_dist['ratio'].round(1),
    marker_color=COLORS_GROUP,
    text=[
        f"{row['game_count']:,}개<br>{row['ratio']:.1f}%"
        for _, row in silence_dist.iterrows()
    ],
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='<b>%{x}</b><br>게임 수: %{customdata:,}<br>비율: %{y:.1f}%<extra></extra>',
    customdata=silence_dist['game_count'],
))

fig.update_layout(
    title=(
        '무반응 그룹 — 지원 언어 수 구간별 분포<br>'
        f'<sub>무반응 그룹 {len(df_silence):,}개 / 리뷰 0~9개</sub>'
    ),
    xaxis_title='지원 언어 수 구간',
    yaxis_title='비율 (%)',
    yaxis=dict(range=[0, silence_dist['ratio'].max() * 1.25]),
    height=440,
    plot_bgcolor='#FAFAFA',
    showlegend=False,
)

fig.show()

**해석:** 무반응 그룹에서 영어 단독 지원 게임이 절반 이상을 차지하며, 5개 이상 다국어 지원 게임 비율은 낮다.
Steam이 언어 지원을 핵심 노출 요소로 명시하고 있음을 감안하면, 언어 지원 범위가 협소한 게임은
해당 언어권 유저에게 노출될 기회 자체가 적어 무반응으로 이어지는 구조적 원인 중 하나일 수 있다.

## 3. 첫 반응군 언어 지원 분포

In [7]:
first_dist = (
    df_first.groupby('lang_group', observed=True)
    .agg(game_count=('appid', 'count'))
    .reset_index()
)
first_dist['ratio'] = first_dist['game_count'] / first_dist['game_count'].sum() * 100

display(
    first_dist.rename(columns={
        'lang_group': '언어 지원 구간',
        'game_count': '게임 수',
        'ratio': '비율 (%)'
    }).round(1).set_index('언어 지원 구간')
)

pct_1  = first_dist.loc[first_dist['lang_group'] == '1개 (영어 단독)', 'ratio'].values[0]
pct_5p = first_dist.loc[first_dist['lang_group'] == '5개 이상', 'ratio'].values[0]
print(f'\n영어 단독 비율  : {pct_1:.1f}%')
print(f'5개 이상 비율   : {pct_5p:.1f}%')

,게임 수,비율 (%)
언어 지원 구간,,
1개 (영어 단독),1945,41.4
2~4개,1305,27.8
5개 이상,1446,30.8



영어 단독 비율  : 41.4%
5개 이상 비율   : 30.8%


**해석:** 첫 반응군에서 영어 단독 지원 비율은 41.4%로 무반응 그룹(56.7%)보다 15.3%p 낮다. 5개 이상 지원 비율은 30.8%로 무반응 대비 10.9%p 높아, 다국어 지원이 첫 반응 달성과 연관되는 경향이 확인된다.

In [8]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=first_dist['lang_group'].astype(str),
    y=first_dist['ratio'].round(1),
    marker_color=COLORS_GROUP,
    text=[
        f"{row['game_count']:,}개<br>{row['ratio']:.1f}%"
        for _, row in first_dist.iterrows()
    ],
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='<b>%{x}</b><br>게임 수: %{customdata:,}<br>비율: %{y:.1f}%<extra></extra>',
    customdata=first_dist['game_count'],
))

fig.update_layout(
    title=(
        '첫 반응군 — 지원 언어 수 구간별 분포<br>'
        f'<sub>첫 반응군 {len(df_first):,}개 / 리뷰 10~49개</sub>'
    ),
    xaxis_title='지원 언어 수 구간',
    yaxis_title='비율 (%)',
    yaxis=dict(range=[0, first_dist['ratio'].max() * 1.25]),
    height=440,
    plot_bgcolor='#FAFAFA',
    showlegend=False,
)

fig.show()

**해석:** 첫 반응군에서는 무반응 그룹 대비 5개 이상 다국어 지원 게임의 비율이 높고,
영어 단독 지원 비율이 상대적으로 낮다.
언어 지원 수가 많을수록 해당 언어권 유저에게 노출될 기회가 넓어지고,
이것이 첫 반응 달성 가능성을 높이는 방향으로 작용함을 시사한다.

## 4. 두 그룹 비교 및 카이제곱 검정

In [9]:
compare = pd.DataFrame({
    '언어 지원 구간': LANG_LABELS,
    '무반응 그룹': silence_dist['ratio'].round(1).values,
    '첫 반응군':   first_dist['ratio'].round(1).values,
})

fig = go.Figure()

fig.add_trace(go.Bar(
    name='무반응 그룹',
    x=compare['언어 지원 구간'],
    y=compare['무반응 그룹'],
    marker_color=COLOR_SILENCE,
    text=compare['무반응 그룹'].apply(lambda v: f'{v:.1f}%'),
    textposition='outside',
))

fig.add_trace(go.Bar(
    name='첫 반응군',
    x=compare['언어 지원 구간'],
    y=compare['첫 반응군'],
    marker_color=COLOR_RESPONSE,
    text=compare['첫 반응군'].apply(lambda v: f'{v:.1f}%'),
    textposition='outside',
))

fig.update_layout(
    barmode='group',
    title=(
        '언어 지원 구간별 그룹 비율 비교 — 무반응 vs 첫 반응군<br>'
        f'<sub>무반응 {len(df_silence):,}개 / 첫 반응군 {len(df_first):,}개</sub>'
    ),
    xaxis_title='지원 언어 수 구간',
    yaxis_title='그룹 내 비율 (%)',
    yaxis=dict(range=[0, max(compare['무반응 그룹'].max(), compare['첫 반응군'].max()) * 1.2]),
    height=460,
    plot_bgcolor='#FAFAFA',
    legend=dict(x=0.75, y=0.95),
)

fig.show()

**해석:** 두 그룹을 나란히 비교하면 영어 단독 지원 비율이 무반응(56.7%) → 첫 반응군(41.4%)으로 15.3%p 낮아지고, 5개 이상 지원 비율은 19.9% → 30.8%로 10.9%p 높아진다. 언어 지원 폭이 넓은 게임이 첫 반응군에 더 많이 분포하는 경향이 있다.

In [10]:
# 분할표 구성: 행 = 그룹(무반응/첫반응), 열 = 언어 구간(3개)
contingency = pd.DataFrame(
    {
        '무반응 그룹': silence_dist['game_count'].values,
        '첫 반응군':   first_dist['game_count'].values,
    },
    index=LANG_LABELS,
).T

print('분할표 (게임 수):')
display(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f'\n카이제곱 통계량 : {chi2:.2f}')
print(f'자유도          : {dof}')
print(f'p-value         : {p_value:.4e}')
print()
if p_value < 0.05:
    print('✓ 유의수준 5%에서 통계적으로 유의함 — 두 그룹의 언어 지원 분포는 서로 다르다.')
else:
    print('✗ 유의수준 5%에서 통계적으로 유의하지 않음.')

print('\n기댓값 (expected):')
display(pd.DataFrame(expected.round(1), index=contingency.index, columns=contingency.columns))

분할표 (게임 수):


,1개 (영어 단독),2~4개,5개 이상
무반응 그룹,3662,1514,1287
첫 반응군,1945,1305,1446



카이제곱 통계량 : 277.70
자유도          : 2
p-value         : 5.0024e-61

✓ 유의수준 5%에서 통계적으로 유의함 — 두 그룹의 언어 지원 분포는 서로 다르다.

기댓값 (expected):


,1개 (영어 단독),2~4개,5개 이상
무반응 그룹,3247.4,1632.7,1582.9
첫 반응군,2359.6,1186.3,1150.1


**해석:** 카이제곱 검정 결과 두 그룹의 언어 지원 분포 차이는 통계적으로 유의하다(p < 0.001). 단, Cramér's V = 0.16으로 효과 크기가 작아, 언어 지원은 단독 결정 요인이 아닌 장르·가격·태그와 함께 작용하는 하나의 출시 조건으로 해석해야 한다.

In [11]:
# 효과 크기: Cramér's V
n = contingency.values.sum()
k = min(contingency.shape) - 1
cramers_v = (chi2 / (n * k)) ** 0.5

print(f"Cramér's V : {cramers_v:.4f}")
if cramers_v < 0.1:
    strength = '매우 약함'
elif cramers_v < 0.3:
    strength = '약함'
elif cramers_v < 0.5:
    strength = '중간'
else:
    strength = '강함'
print(f'효과 크기  : {strength}')

Cramér's V : 0.1578
효과 크기  : 약함


**해석:**

카이제곱 검정 결과, 두 그룹의 언어 지원 분포 차이는 통계적으로 유의하다 (p < 0.05).
즉 무반응 그룹과 첫 반응군 사이의 언어 지원 비율 차이는 우연에 의한 것이 아니다.

Cramér's V로 측정한 효과 크기는 비교적 작지만, 언어 지원이 다른 많은 요인들과 복합적으로 작용함을 고려하면
독립 변수로서 유의미한 관련성을 가진다고 볼 수 있다.

**발표 인사이트:**
- 무반응 그룹은 영어 단독 지원 비율이 높고, 첫 반응군은 5개 이상 다국어 지원 비율이 높다.
- Steam이 언어 지원을 핵심 노출 요소로 명시한 것과 일치하는 데이터 패턴이다.
- 언어 지원 확대는 개발 완료 후 추가하기 어렵고 출시 전 결정해야 하는 사항이므로, 전략적 선택의 우선순위에 포함해야 한다.